# Step 01. Planning Pipeline 전체 구조

> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

자율주행 소프트웨어에서 Planning 모듈의 위치와 Global / Local Planner의 역할을 이해합니다.  
Day 1(Global Planning) → Day 2(Local Planning) 전체 흐름의 출발점입니다.

## 학습 목표

1. 자율주행 소프트웨어 스택에서 Planning 모듈의 위치 파악
2. Global Planner와 Local Planner의 역할, 입출력, 갱신 주기 차이 이해
3. Day 1 커리큘럼 (Dijkstra → A* → Hybrid A*) 흐름 파악
4. 전체 Planning 파이프라인 흐름 이해

> 참고: matplotlib 이미지에는 한글이 깨지므로 그림 라벨은 모두 영어로 표기합니다. 한국어 설명은 `print()` 출력으로 제공합니다.

In [ ]:
%matplotlib inline
import sys, importlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

libs = ['numpy', 'matplotlib', 'scipy']
print('라이브러리 버전 확인')
print('-' * 35)
for lib in libs:
    try:
        m = importlib.import_module(lib)
        print(f'  {lib:<14} {m.__version__}')
    except ImportError:
        print(f'  {lib:<14} 미설치')
print(f'  Python         {sys.version.split()[0]}')
print()
print('환경 준비 완료 — Day 1 실습을 시작합니다')


## 1. 자율주행 소프트웨어 스택

### 이론 배경

자율주행 시스템은 크게 5단계 파이프라인으로 구성됩니다.

```
Perception  →  Prediction  →  Planning  →  Control  →  Actuator
  (인식)        (예측)         (계획)       (제어)       (구동)
```

**Planning** 모듈은 *어디로 갈지*를 결정합니다.

- **입력**: HD Map(도로 구조 정보) + Prediction(주변 차량/보행자 예측 위치)
- **출력**: Trajectory — 시간에 따른 위치·속도·방향 시퀀스

Planning은 Perception/Prediction 결과를 *목적지까지의 주행 계획*으로 변환하는 역할입니다.

In [ ]:
# Autonomous driving software stack (English labels only — Korean breaks in images)
fig, ax = plt.subplots(figsize=(13, 2.6))
ax.axis('off')

modules = ['Perception', 'Prediction', 'Planning', 'Control', 'Actuator']
subtxt  = ['sensing',    'forecasting','decision',  'command', 'actuation']
colors  = ['#4C72B0', '#4C72B0', '#C44E52', '#4C72B0', '#4C72B0']  # Planning highlighted

BOX_W, BOX_H, GAP = 1.9, 1.0, 0.7
pitch = BOX_W + GAP
x_pos = [i * pitch for i in range(len(modules))]

for x, name, sub, color in zip(x_pos, modules, subtxt, colors):
    ax.add_patch(mpatches.FancyBboxPatch(
        (x, 0), BOX_W, BOX_H,
        boxstyle='round,pad=0.02,rounding_size=0.12',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.95))
    ax.text(x + BOX_W/2, 0.62, name, ha='center', va='center',
            color='white', fontsize=11, fontweight='bold')
    ax.text(x + BOX_W/2, 0.30, sub, ha='center', va='center',
            color='white', fontsize=8, alpha=0.9, style='italic')

# Arrows between boxes (from right edge of a box to left edge of the next)
for x in x_pos[:-1]:
    ax.annotate('', xy=(x + pitch, BOX_H/2), xytext=(x + BOX_W, BOX_H/2),
                arrowprops=dict(arrowstyle='-|>', color='#555555', lw=2,
                                mutation_scale=18))

ax.set_xlim(-0.4, x_pos[-1] + BOX_W + 0.4)
ax.set_ylim(-0.5, 1.5)
ax.set_title('Autonomous Driving Software Stack   (Planning = red)',
             fontsize=13, pad=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Planning 모듈의 입출력')
print('  Input : HD Map (도로 정보) + Prediction 결과 (장애물 예측 위치)')
print('  Output: Trajectory (위치 + 속도 + 시간 시퀀스)')


## 2. Global Planner vs Local Planner

### 이론 배경

Planning은 역할에 따라 두 계층으로 나뉩니다.

| 구분 | Global Planner | Local Planner |
|------|---------------|---------------|
| 입력 | HD Map + 목적지 | Global Path + 실시간 센서 |
| 출력 | Global Path (경유점 목록) | 주행 가능한 Trajectory |
| 범위 | 맵 전체 (수 km) | 수십 m 앞 |
| 빈도 | 저빈도 (경로 변경 시) | 고빈도 (10~20 Hz) |
| 알고리즘 | Dijkstra, A*, Hybrid A* | Frenet, RRT, MPC |

**이번 Day 1은 Global Planner**, **Day 2는 Local Planner**를 다룹니다.

In [ ]:
# Global vs Local Planner — relative characteristics (English only)
categories  = ['Planning\nRange', 'Update\nRate', 'Obstacle\nReactivity', 'Map\nDependency']
global_vals = [5, 1, 2, 5]   # normalized 1-5
local_vals  = [1, 5, 5, 2]

y = np.arange(len(categories))
h = 0.36
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(y + h/2, global_vals, height=h, color='#4C72B0', alpha=0.9,
        label='Global Planner (Dijkstra, A*)')
ax.barh(y - h/2, local_vals,  height=h, color='#DD8452', alpha=0.9,
        label='Local Planner (Frenet, RRT)')

ax.set_yticks(y)
ax.set_yticklabels([c.replace('\n', ' ') for c in categories], fontsize=10)
ax.invert_yaxis()
ax.set_xlim(0, 5.8)
ax.set_xlabel('Relative score (1 = low, 5 = high)')
ax.set_title('Global Planner vs Local Planner', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)

# annotate bar values
for yi, gv, lv in zip(y, global_vals, local_vals):
    ax.text(gv + 0.1, yi + h/2, str(gv), va='center', fontsize=8, color='#33506e')
    ax.text(lv + 0.1, yi - h/2, str(lv), va='center', fontsize=8, color='#9c5a30')

plt.tight_layout()
plt.show()

print('특성 비교 요약')
print(f'  {"항목":<14}  {"Global Planner":<22}  Local Planner')
print('  ' + '-' * 60)
rows = [
    ('입력',         'HD Map + 목적지',           'Global Path + 센서'),
    ('출력',         'Global Path',               'Trajectory (위치+속도)'),
    ('갱신 주기',    '저빈도 (경로 변경 시)',      '고빈도 (10~20 Hz)'),
    ('탐색 범위',    '맵 전체',                   '수십 m 앞'),
    ('대표 알고리즘','Dijkstra / A*',             'Frenet / RRT'),
]
for item, g, l in rows:
    print(f'  {item:<14}  {g:<22}  {l}')


## 3. Day 1 실습 커리큘럼

### 이론 배경

Day 1에서는 알고리즘 복잡도를 점진적으로 높여가며 Global Planner를 학습합니다.

```
Dijkstra  →  A*  →  Hybrid A*
  (기준)    (+Heuristic)  (+연속 상태 공간)
```

- **Dijkstra** (Step 02): Grid 탐색의 기본 원리. 최소 비용 우선 확장. 방사형 탐색.
- **A\*** (Step 03): Heuristic `h(n)` 추가 → 목표 방향 집중 → 탐색 효율 대폭 향상.
- **Hybrid A\*** (Step 04): 상태 공간을 (x,y) 격자 → (x,y,yaw) 연속으로 확장. Bicycle Model로 차량 기구학 반영 → 실제 차량이 주행 가능한 경로 생성.

In [ ]:
# Day 1 curriculum as a progression diagram (English labels)
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.axis('off')

steps = [
    ('Step 01', 'Architecture',  'pipeline overview',          '#7f8c8d'),
    ('Step 02', 'Dijkstra',      'grid + min-cost search',     '#2980b9'),
    ('Step 03', 'A*',            '+ heuristic -> faster',      '#27ae60'),
    ('Step 04', 'Hybrid A*',     '+ (x, y, yaw) continuous',   '#c0392b'),
]
BOX_W, BOX_H, GAP = 2.6, 1.4, 0.9
pitch = BOX_W + GAP
x_pos = [i * pitch for i in range(len(steps))]

for x, (sid, title, desc, color) in zip(x_pos, steps):
    ax.add_patch(mpatches.FancyBboxPatch(
        (x, 0), BOX_W, BOX_H,
        boxstyle='round,pad=0.02,rounding_size=0.12',
        facecolor=color, edgecolor='white', linewidth=2, alpha=0.95))
    ax.text(x + BOX_W/2, 1.05, sid,   ha='center', color='white', fontsize=9,  alpha=0.9)
    ax.text(x + BOX_W/2, 0.68, title, ha='center', color='white', fontsize=12, fontweight='bold')
    ax.text(x + BOX_W/2, 0.30, desc,  ha='center', color='white', fontsize=7.5, style='italic')

for x in x_pos[:-1]:
    ax.annotate('', xy=(x + pitch, BOX_H/2), xytext=(x + BOX_W, BOX_H/2),
                arrowprops=dict(arrowstyle='-|>', color='#555555', lw=2,
                                mutation_scale=18))

ax.set_xlim(-0.4, x_pos[-1] + BOX_W + 0.4)
ax.set_ylim(-0.4, 1.9)
ax.set_title('Day 1 — Increasing Algorithm Complexity',
             fontsize=13, pad=10, fontweight='bold')
plt.tight_layout()
plt.show()

print('Day 1 — Global Planning 실습 커리큘럼')
print('=' * 56)
curriculum = [
    ('Step 01', 'Planning Pipeline 전체 구조', '지금 여기'),
    ('Step 02', 'Dijkstra',                    'Grid + 최소 비용 탐색 (기준선)'),
    ('Step 03', 'A*',                          'Heuristic 추가 -> 탐색 효율 향상'),
    ('Step 04', 'Hybrid A*',                   '(x,y,yaw) 연속 상태 + Bicycle Model'),
]
for sid, title, desc in curriculum:
    print(f'  {sid}  {title:<28}  {desc}')
print('=' * 56)


## 4. Planning 흐름 요약

### 이론 배경

Global Planner와 Local Planner는 계층적으로 협력합니다.

- **Global Planner**가 먼저 전체 경로(Global Path)를 생성합니다.
- **Local Planner**는 Global Path를 따라가면서 실시간 장애물을 피하는 Trajectory를 생성합니다.
- Global Path 없이 Local Planner만 있으면 목적지 방향을 알 수 없어 큰 우회가 발생합니다.

이 두 계층의 협력이 자율주행의 '어디로 갈지'를 결정하는 핵심 구조입니다.

In [ ]:
# End-to-end planning flow (English labels only)
fig, ax = plt.subplots(figsize=(11.5, 5.5))
ax.axis('off')

def box(x, y, w, h, label, color, fs=9):
    ax.add_patch(mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.02,rounding_size=0.08',
        facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.95))
    ax.text(x, y, label, ha='center', va='center',
            color='white', fontsize=fs, fontweight='bold')

GREEN, BLUE, RED, PURPLE, GRAY = '#2ecc71', '#3498db', '#e74c3c', '#9b59b6', '#95a5a6'

# main pipeline column (x = 5)
box(5, 5.2, 2.6, 0.7, 'Start  (sx, sy)',  GREEN)
box(5, 4.0, 2.6, 0.8, 'Global Planner',   RED, fs=10)
box(5, 2.8, 2.6, 0.7, 'Global Path',      GRAY)
box(5, 1.6, 2.6, 0.8, 'Local Planner',    RED, fs=10)
box(5, 0.4, 2.6, 0.7, 'Trajectory',       GRAY)
box(5, -0.8,2.6, 0.8, 'Controller',       PURPLE, fs=10)
box(5, -2.0,2.6, 0.7, 'Goal  (gx, gy)',   GREEN)

# side inputs
box(1.2, 4.0, 2.2, 0.8, 'HD Map\n(road network)', BLUE, fs=8)
box(1.2, 1.6, 2.2, 0.8, 'Sensors\n(obstacles)',   BLUE, fs=8)

# vertical pipeline arrows
ys = [(5.2, 4.4), (3.6, 3.15), (2.45, 2.0), (1.2, 0.75), (0.05, -0.4), (-1.2, -1.65)]
for y1, y2 in ys:
    ax.annotate('', xy=(5, y2), xytext=(5, y1),
                arrowprops=dict(arrowstyle='-|>', color='#555555', lw=2, mutation_scale=16))

# side input arrows
ax.annotate('', xy=(3.7, 4.0), xytext=(2.3, 4.0),
            arrowprops=dict(arrowstyle='-|>', color=BLUE, lw=1.8, mutation_scale=14))
ax.annotate('', xy=(3.7, 1.6), xytext=(2.3, 1.6),
            arrowprops=dict(arrowstyle='-|>', color=BLUE, lw=1.8, mutation_scale=14))

# edge labels
ax.text(5.25, 3.4, 'path',       fontsize=7.5, color='#444', ha='left', style='italic')
ax.text(5.25, 1.0, 'trajectory', fontsize=7.5, color='#444', ha='left', style='italic')

ax.set_xlim(-0.2, 7.2)
ax.set_ylim(-2.6, 5.8)
ax.set_title('Planning Pipeline — Global / Local Cooperation',
             fontsize=13, pad=8, fontweight='bold')
plt.tight_layout()
plt.show()

print('Global Path 없이 Local Planner만 있으면?')
print('  -> 목적지까지 전체 경로를 모르므로 큰 우회나 막힘 발생')
print('  -> Global Path 가 "어느 방향으로 가야 하는지" 큰 그림을 제공')
print()
print('다음 단계: Step 02에서 Dijkstra로 Global Path를 탐색합니다')
